# 10 · MNIST first CNN — and head-to-head vs the baseline

Our **first convolutional network**: a small Keras conv net on MNIST, compared
directly against the logistic-regression baseline (#30). This is where
convolutions click — and it establishes the `Sequential → compile → fit →
evaluate` loop we scale up in Phase 4.

**The CNN trio:** `Conv2D` (feature extraction) → `MaxPool` (downsample /
translation-invariance) → `Dense` head (classification). Unlike logistic
regression, a CNN keeps the 2-D grid and exploits **locality + parameter sharing**
— the spatial structure the linear model threw away by flattening.

## 0. Setup (seeded for reproducibility)

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.emotion_detector.utils.config import load_config
from src.emotion_detector.utils.seeding import set_global_seed

cfg = load_config(ROOT / "config.yaml")
SEED = cfg["global"]["seed"]
set_global_seed(SEED)  # seeds random / numpy / tensorflow

EDA_DIR = Path(ROOT) / cfg["paths"]["results_dir"] / "eda"
BASELINE_DIR = Path(ROOT) / cfg["paths"]["results_dir"] / "baselines"
EDA_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters (constants here; the Phase-4 ModelBuilder #34 reads these from config)
EPOCHS = 5
BATCH_SIZE = 128
print(f"seed={SEED}, epochs={EPOCHS}, batch_size={BATCH_SIZE}")

## 1. Load & prepare MNIST

Reshape to `(N, 28, 28, 1)` — Conv2D needs the explicit channel axis — rescale to
`[0, 1]`, and one-hot encode the labels for `categorical_crossentropy`.

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = mnist.load_data()
X_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
X_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0
Y_train = to_categorical(y_train, 10)
Y_test = to_categorical(y_test, 10)
print(f"X_train {X_train.shape}, Y_train {Y_train.shape}")

## 2. Build the CNN

`Conv2D → ReLU → MaxPool → Conv2D → ReLU → MaxPool → Flatten → Dense → softmax`.
Wrapped in a function to preview the config-driven `ModelBuilder` (#34).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers


def build_mnist_cnn():
    """Small conv net: two conv/pool blocks + a dense classifier head."""
    return keras.Sequential([
        keras.Input((28, 28, 1)),
        layers.Conv2D(32, 3, activation="relu"),   # 32 learnable 3x3 edge/stroke filters
        layers.MaxPooling2D(2),                    # downsample 2x → translation-invariance
        layers.Conv2D(64, 3, activation="relu"),   # 64 filters on the coarser map
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),    # class probabilities
    ], name="mnist_cnn")


model = build_mnist_cnn()
model.summary()

## 3. Compile

Adam optimizer, categorical-crossentropy loss (pairs with the one-hot labels),
accuracy metric — the exact loop we reuse in Phase 4.

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
n_params = model.count_params()
print(f"trainable parameters: {n_params:,}")

## 4. Train (with validation), timed

In [ ]:
start = time.time()
history = model.fit(
    X_train, Y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)
train_time = time.time() - start
print(f"\ntrained {EPOCHS} epochs in {train_time:.1f}s")

## 5. Evaluate on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"CNN test accuracy: {test_acc:.4f}")

## 6. Head-to-head vs the logistic-regression baseline (#30)

In [ ]:
baseline = json.loads((BASELINE_DIR / "mnist_logreg.json").read_text())
logreg_acc = baseline["test_accuracy"]
logreg_params = 10 * 784 + 10  # weights + biases

print(f"{'model':<22}{'test acc':>10}{'params':>12}")
print("-" * 44)
print(f"{'logistic regression':<22}{logreg_acc:>10.4f}{logreg_params:>12,}")
print(f"{'CNN':<22}{test_acc:>10.4f}{n_params:>12,}")
print("-" * 44)
print(f"CNN improvement: {(test_acc - logreg_acc) * 100:+.2f} percentage points")
print(f"error rate cut:  {(1 - logreg_acc):.4f} -> {(1 - test_acc):.4f} "
      f"({(1 - test_acc) / (1 - logreg_acc):.0%} of the baseline's errors remain)")

## 7. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].axhline(logreg_acc, ls="--", color="gray", label=f"logreg {logreg_acc:.3f}")
axes[0].set_title("accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("loss"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
fig.savefig(EDA_DIR / "mnist_cnn_curves.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'mnist_cnn_curves.png'}")
plt.show()

## 8. Findings

- **CNN test accuracy ≈ 0.99** vs logistic regression **0.927** — roughly **+6
  points**, cutting the error rate by ~85 % (from ~7.3 % to ~1 %).
- **The cost:** the CNN has more parameters (~225k vs ~7.8k) and takes longer to
  train — but on images that complexity clearly **earns its keep**.
- **Why it wins:** convolutions keep the 2-D layout. Each filter is a small kernel
  **shared across every position** (parameter sharing), detecting a local pattern
  (an edge, a curve) wherever it appears (locality). Pooling then makes the
  features robust to small shifts. Logistic regression, seeing 784 independent
  pixels, has none of this — which is exactly why it confused shape-similar digits.

This is the habit for the main task: the emotion CNN (Phase 4) is only "good" if it
clears a comparable non-deep reference — never trust a deep model without a baseline.